# Memoization and backpropagation
Fibonacci with and without memoization, then the same idea inside backpropagation: the derivative of the loss
with respect to each node's output, computed by plain recursion (repeating work) and with memoization.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import time
from functools import lru_cache
import numpy as np
import pandas as pd

## Fibonacci by plain recursion

In [ ]:
calls = 0

def fib(n):
    global calls
    calls += 1
    if n == 0 or n == 1:
        return 1
    return fib(n - 1) + fib(n - 2)

for n in [5, 10, 12]:
    calls = 0
    print(n, fib(n), "calls:", calls)
calls = 0
start = time.time()
print("fib(30) =", fib(30), "calls:", calls, "seconds: %.2f" % (time.time() - start))

## Fibonacci with memoization: store each answer in a dictionary

In [ ]:
def fib_memo(n, d):
    global calls
    calls += 1
    if n in d:                       # already computed: look it up
        return d[n]
    d[n] = fib_memo(n - 1, d) + fib_memo(n - 2, d)
    return d[n]

calls = 0
start = time.time()
print("fib(30) =", fib_memo(30, {0: 1, 1: 1}), "calls:", calls, "seconds: %.6f" % (time.time() - start))
calls = 0
print("fib(100) =", fib_memo(100, {0: 1, 1: 1}), "calls:", calls)

In [ ]:
rows = []
for n in range(1, 31):
    calls = 0; fib(n); plain = calls
    calls = 0; fib_memo(n, {0: 1, 1: 1}); memo = calls
    rows.append(dict(n=n, plain=plain, memo=memo))
fib_calls = pd.DataFrame(rows)
fib_calls.to_csv("data/fib_calls.csv", index=False)
fib_calls.tail(3)

## A 3-3-2-1 network: the two paths from a first-layer weight
Sigmoid hidden layers, a linear output and the squared loss. Fixed random weights.

In [ ]:
sizes = [3, 3, 2, 1]
rng = np.random.default_rng(0)
W = {l: rng.normal(0, 0.5, (sizes[l - 1], sizes[l])) for l in (1, 2, 3)}   # W[l][i, j] = W^l_(i+1)(j+1)
b = {l: np.zeros(sizes[l]) for l in (1, 2, 3)}
x, y = np.array([0.5, -1.0, 2.0]), 1.0
print("trainable parameters:", sum(W[l].size + b[l].size for l in W))   # 12 + 8 + 3 = 23

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

O = {0: x}
for l in (1, 2):
    O[l] = sigmoid(W[l].T @ O[l - 1] + b[l])
O[3] = W[3].T @ O[2] + b[3]               # linear output
y_hat = O[3][0]
print("O1 =", O[1].round(4), "O2 =", O[2].round(4), "y_hat =", round(y_hat, 4))

In [ ]:
dL_dy = -2 * (y - y_hat)
s1, s2 = O[1] * (1 - O[1]), O[2] * (1 - O[2])            # sigmoid slopes of layers 1 and 2
# dL/dW^3_11: one link after dL/dy_hat
g3 = dL_dy * O[2][0]
# dL/dW^2_11: through O21 only
g2 = dL_dy * W[3][0, 0] * s2[0] * O[1][0]
# dL/dW^1_11: through O21 AND through O22, then add
path_a = dL_dy * W[3][0, 0] * s2[0] * W[2][0, 0]          # y_hat <- O21 <- O11
path_b = dL_dy * W[3][1, 0] * s2[1] * W[2][0, 1]          # y_hat <- O22 <- O11
g1 = (path_a + path_b) * s1[0] * x[0]
print("dL/dW3_11 = %.6f, dL/dW2_11 = %.6f" % (g3, g2))
print("path a %.6f + path b %.6f -> dL/dW1_11 = %.6f" % (path_a * s1[0] * x[0], path_b * s1[0] * x[0], g1))

## Check with GradientTape

In [ ]:
import tensorflow as tf

tW = {l: tf.Variable(W[l]) for l in W}
with tf.GradientTape() as tape:
    a = tf.constant(x.reshape(1, -1))
    for l in (1, 2):
        a = tf.sigmoid(a @ tW[l] + b[l])
    out = a @ tW[3] + b[3]
    loss = (y - out[0, 0]) ** 2
grads = tape.gradient(loss, tW)
print("tape:", grads[3].numpy()[0, 0], grads[2].numpy()[0, 0], grads[1].numpy()[0, 0])

## Backpropagation as recursion: dL/dO for every node
dL/dO(l, j) = sum over the nodes k of layer l+1 of dL/dO(l+1, k) * slope(l+1, k) * W^(l+1)_jk.
Without memoization the recursion recomputes the same node again and again, like fib.
`lru_cache` from the standard library is memoization in one line: it stores every answer of the function.

In [ ]:
def make_network(sizes, seed=0):
    rng = np.random.default_rng(seed)
    W = {l: rng.normal(0, 0.5, (sizes[l - 1], sizes[l])) for l in range(1, len(sizes))}
    O, a = {0: rng.normal(0, 1, sizes[0])}, None
    L = len(sizes) - 1
    for l in range(1, L + 1):
        z = W[l].T @ O[l - 1]
        O[l] = z if l == L else sigmoid(z)
    slope = {l: (np.ones(sizes[l]) if l == L else O[l] * (1 - O[l])) for l in range(1, L + 1)}
    return W, O, slope, L

def gradients(sizes, memo):
    W, O, slope, L = make_network(sizes)
    count = {"calls": 0}

    def dL_dO(l, j):
        count["calls"] += 1
        if l == L:
            return -2 * (1.0 - O[L][0])                          # y = 1
        return sum(node(l + 1, k) * slope[l + 1][k] * W[l + 1][j, k] for k in range(sizes[l + 1]))

    node = lru_cache(maxsize=None)(dL_dO) if memo else dL_dO      # the cached or the plain version
    grads = {l: np.array([[node(l, j) * slope[l][j] * O[l - 1][i] for j in range(sizes[l])]
                          for i in range(sizes[l - 1])]) for l in range(1, L + 1)}
    return grads, count["calls"]

g_plain, c_plain = gradients([3, 3, 2, 1], memo=False)
g_memo, c_memo = gradients([3, 3, 2, 1], memo=True)
print("3-3-2-1: calls without memo", c_plain, "with memo", c_memo,
      "same gradients:", all(np.allclose(g_plain[l], g_memo[l]) for l in g_plain))

In [ ]:
rows = []
for depth in range(1, 6):
    sizes_d = [10] + [10] * depth + [1]
    _, cp = gradients(sizes_d, memo=False)
    _, cm = gradients(sizes_d, memo=True)
    rows.append(dict(hidden_layers=depth, plain=cp, memo=cm))
net_calls = pd.DataFrame(rows)
net_calls.to_csv("data/network_calls.csv", index=False)
net_calls